# Structural Validator (Goal 0)

## What this notebook does

Before we simulate anything, a rocket design has to be *possible* — structurally valid in the KSP sense.
This notebook builds the validator that enforces that.

## The rocket format

For now, a rocket is a plain Python dict. This keeps things inspectable and printable.
A later notebook (`rocket_representation.ipynb`) will build a class that produces this same format —
so the validator doesn't need to change when that happens.

## What makes a rocket structurally valid?

A valid design must satisfy all of the following:

- **One connected part tree** — all parts form a single connected graph, no orphans
- **Exactly one root part** — one part has no parent (the top of the stack)
- **At least one command source** — a part with `is_command = True`
- **At least one engine** — a part with engine data
- **Propellant compatibility** — every engine has the resources it needs available somewhere in the design
- **Valid part references** — every `type` field refers to a real part in the parts library
- **Valid attachment nodes** — parent/child attachments reference nodes that actually exist on those parts
- **No cycles** — the part tree has no loops
- **Valid stage references** — stage numbers are non-negative integers

We'll build and test one check at a time.

In [ ]:
### setup block

from pathlib import Path
import json

parts_json_file = Path('../data/parts_library.json')
with open(parts_json_file) as f:
    parts_library = json.load(f)

parts_by_name = {p['name']: p for  p in parts_library }

rocket_example = {
    "parts": [
        {"id": "pod_0",  "type": "mk1-3pod", "parent": None},
        {"id": "tank_0", "type": "fuelTank", "parent": "pod_0", "attach_node": "bottom"  },
        {"id": "eng_0",  "type": "liquidEngine", "parent": "tank_0", "attach_node": "bottom"},
    ],
    "stages": {"eng_0": 0}
}

for part in rocket_example["parts"]:
    print(part["parent"])


first, load in json parts list, inspect format, and create dict of categories/parts. then a function to check against

In [ ]:
def check_part_call(part: str,
                    parts_by_name_dict: dict):
    if part not in parts_by_name_dict:
        return False
    else:
        return True

print(check_part_call('liquidEngine', parts_by_name))
print(check_part_call('nope', parts_by_name))


check minimal parts for rocket. built individual checks and then final helper

In [ ]:
def check_single_root(rocket_dict: dict):
    num_no_parents = 0
    for part in rocket_dict['parts']:
        if part['parent'] is None:
            num_no_parents += 1

    if num_no_parents != 1:
        return False
    return True

def check_has_command(rocket_dict: dict,
                      parts_dict: dict):
    command_count = 0
    for part in rocket_dict['parts']:
        part_name = part['type']
        part_info = parts_dict[part_name]
        is_command = part_info['is_command']
        if is_command:
            command_count += 1
    if command_count > 0:
        return True
    return False

def check_has_engine(rocket_dict: dict,
                     parts_dict: dict):
    engine_count = 0
    for part in rocket_dict['parts']:
        part_name = part['type']
        part_info = parts_dict[part_name]
        is_engine = part_info['engine']
        if is_engine:
            engine_count += 1
    if engine_count > 0:
        return True
    return False


def has_minimal_structure(rocket_dict: dict,
                          parts_dict: dict):
    root_check = check_single_root(rocket_dict)
    command_check = check_has_command(rocket_dict, parts_dict)
    engine_check = check_has_engine(rocket_dict, parts_dict)
    valid_struct = all([root_check, command_check, engine_check])

    return valid_struct


print(check_single_root(rocket_example))
print(check_has_command(rocket_example, parts_by_name))
print(check_has_engine(rocket_example, parts_by_name))
print(has_minimal_structure(rocket_example, parts_by_name))


check ship structure

In [ ]:
def check_graph_connections(rocket_dict:dict,
                            parts_dict: dict,
                            verbose: bool = False):
    all_parts = {part['id'] for part in rocket_dict['parts']}
    root_id = next(part['id'] for part in rocket_dict['parts'] if part['parent'] is None)
    children = {part['id']: [] for part in rocket_dict['parts']}
    for part in rocket_dict['parts']:
        if part['parent'] is not None:
            children[part['parent']].append(part['id'])

    queue = [root_id]
    visited = set()

    while queue:
        current = queue.pop(0)
        visited.add(current)
        for child in children[current]: ##### Note: this is a check of circularity because KSP enforces a linear graph structure on its rockets. This would not exist on a real rocket that has various circular systems
            if child in visited:
                return False
            else:
                queue.append(child)

    if visited != all_parts:
        return False
    if verbose:
        print(visited)
        return True
    return True

check_graph_connections(rocket_example, parts_by_name)


check propellant compatibility


In [ ]:

def check_propellant(rocket_dict: dict,
                     parts_dict: dict):
    available_resources = set()
    for part in rocket_dict['parts']:
        part_info = parts_dict[part['type']]
        if part_info['resources']:
            available_resources.update(part_info['resources'].keys())

    needed_propellants = set()
    for part in rocket_dict['parts']:
        if parts_dict[part['type']]['engine'] is not None:
            needed_propellants.update(parts_dict[part['type']]['engine']['propellants'].keys())
    if needed_propellants.issubset(available_resources):
        return True
    return False

check_propellant(rocket_example, parts_by_name)

make sure that all parts that are staged are in the rocket dict

In [ ]:
def check_staging(rocket_dict: dict):

    all_parts = {part['id'] for part in rocket_dict['parts']}

    stages = set()
    for name, stage  in rocket_dict['stages'].items():
        if not isinstance(stage, int) or stage < 0:
            return False
        stages.add(name)

    if stages.issubset(all_parts):
        return True
    return False

print(check_staging(rocket_example))

check node connections

In [ ]:
def check_valid_nodes(rocket_dict: dict,
                      parts_dict: dict):
    id_to_type = {p['id']: p['type'] for p in rocket_dict['parts']}


    for part in rocket_dict['parts']:
        if part['parent'] == None:
            continue
        parent_type = id_to_type[part['parent']]
        if not isinstance(part['attach_node'], str):
            return False
        if part['attach_node'] not in parts_dict[parent_type]['nodes'].keys():
            return False
    return True

check_valid_nodes(rocket_example, parts_by_name)

make the final wrapper

In [ ]:
def validate_rocket(rocket_dict: dict,
                    parts_dict: dict,
                    verbose: bool = False):
    for part in rocket_dict['parts']:
        if not check_part_call(part['type'], parts_dict):
            if verbose:
                print(f"FAIL: unknown part type '{part['type']}")
            return False
    checks = [
          (has_minimal_structure(rocket_dict, parts_dict),  "missing root, command, or engine"),
          (check_graph_connections(rocket_dict, parts_dict), "part tree disconnected or has cycles"),
          (check_propellant(rocket_dict, parts_dict),        "propellant incompatibility"),
          (check_staging(rocket_dict),                       "invalid stage references"),
          (check_valid_nodes(rocket_dict, parts_dict),       "invalid attachment nodes"),
      ]
    for result, message in checks:
          if not result:
              if verbose:
                  print(f"FAIL: {message}")
              return False
    return True

validate_rocket(rocket_example, parts_by_name)

adding functions to do CoM and height checks

In [ ]:
from src.config import load_resource_lookup
resource_lookup = load_resource_lookup()

# map each part id to its full part dict for fast lookup
def build_part_lookup(rocket_dict: dict):
    part_lookup = {}

    for part in rocket_dict['parts']:
        part_id = part['id']
        part_lookup[part_id] = part

    return part_lookup

# map each part id to a list of its direct children
def build_children_lookup(rocket_dict: dict):
    children_lookup = {}

    for part in rocket_dict['parts']:
        part_id = part['id']
        children_lookup[part_id] = []

    for part in rocket_dict['parts']:
        if part['parent'] is not None:
            parent_id = part['parent']
            child_id = part['id']

            children_lookup[parent_id].append(child_id)

    return children_lookup

# return the single root part id, or None if the rocket has zero or multiple roots
def get_root_part_id(rocket_dict: dict):
    root_ids = []
    for part in rocket_dict['parts']:
        if part['parent'] is None:
            root_ids.append(part['id'])

    if len(root_ids) != 1:
        return None
    return root_ids[0]

#return the ordered part ids from top to bottom along the main inline chain
def get_inline_stack_ids(rocket_dict: dict):
    children_lookup = build_children_lookup(rocket_dict)
    root_id = get_root_part_id(rocket_dict)

    if root_id is None:
        return None

    stack_ids = []
    current_id = root_id

    while current_id is not None:
        stack_ids.append(current_id)

        children = children_lookup[current_id]

        if len(children) > 1:
            return None

        if len(children) == 0:
            current_id = None
        else:
            current_id = children[0]

    return stack_ids

# estimate part height from the vertical distance between its top and bottom nodes
def get_part_height(part_info: dict):
    nodes = part_info['nodes']

    if 'top' not in nodes or 'bottom' not in nodes:
        return None

    top_position = nodes['top']['pos']
    bottom_position = nodes['bottom']['pos']

    top_y = top_position[1]
    bottom_y = bottom_position[1]

    return abs(top_y - bottom_y)

## prefer bulkhead profile diameter classes, then fall back to inline node sizes
def get_part_diameter_proxy(part_info: dict):
    profile_sizes = {
        'size0': 0.625,
        'size1': 1.25,
        'size1p5': 1.875,
        'size2': 2.5,
        'size3': 3.75,
    }

    bulkhead_profiles = part_info['bulkhead_profiles']
    if bulkhead_profiles is not None:
        profile_diameters = []
        for profile in bulkhead_profiles:
            if profile in profile_sizes:
                profile_diameters.append(profile_sizes[profile])
        if len(profile_diameters) > 0:
            return max(profile_diameters)

    nodes = part_info['nodes']
    sizes = []

    if 'top' in nodes and nodes['top']['size'] > 0:
        sizes.append(nodes['top']['size'])

    if 'bottom' in nodes and nodes['bottom']['size'] > 0:
        sizes.append(nodes['bottom']['size'])

    if len(sizes) == 0:
        return None

    return max(sizes)

# estimate wet part mass as dry mass plus stored resource masst
def get_part_mass_proxy(part_info: dict,
                        resource_lookup: dict):
    total_mass = part_info['mass_t']

    if part_info['resources'] is not None:
        for resource_name, amount in part_info['resources'].items():
            density = resource_lookup[resource_name]['density']
            total_mass += amount * density
    return total_mass

### get center of mass along part axis
def get_part_axial_com(part_info: dict):
    nodes = part_info['nodes']

    if 'top' not in nodes or 'bottom' not in nodes:
        return None

    top_position = nodes['top']['pos']
    bottom_position = nodes['bottom']['pos']

    top_y = top_position[1]
    bottom_y = bottom_position[1]

    midpoint = (top_y + bottom_y) / 2

    if part_info['com_offset'] is not None:
        midpoint += part_info['com_offset'][1]

    return midpoint


#### assemble the geometry metrics into one dict
def compute_geometry_metrics(rocket_dict: dict,
                             parts_dict: dict,
                             resource_lookup: dict):

    part_lookup = build_part_lookup(rocket_dict)
    stack_ids = get_inline_stack_ids(rocket_dict)

    if stack_ids is None:
        return None

    part_heights = []
    part_diameters = []
    part_masses = []
    part_axial_coms = []

    for part_id in stack_ids:
        placed_part = part_lookup[part_id]
        part_type = placed_part['type']
        part_info = parts_dict[part_type]

        height = get_part_height(part_info)
        diameter = get_part_diameter_proxy(part_info)
        mass = get_part_mass_proxy(part_info, resource_lookup)
        axial_com = get_part_axial_com(part_info)

        if height is None or diameter is None or axial_com is None:
            return None

        part_heights.append(height)
        part_diameters.append(diameter)
        part_masses.append(mass)
        part_axial_coms.append(axial_com)

    total_height = sum(part_heights)

    if total_height <= 0:
        return None

    diameter_transitions = []

    for item in range(len(part_diameters) - 1):
        current_diameter = part_diameters[item]
        next_diameter = part_diameters[item + 1]

        if current_diameter != next_diameter:
            transition_size = abs(next_diameter - current_diameter)
            diameter_transitions.append(transition_size)

    diameter_transition_count = len(diameter_transitions)
    diameter_transition_severity = sum(diameter_transitions)

    max_diameter = max(part_diameters)
    if max_diameter <= 0:
        return None

    slenderness_ratio = total_height / max_diameter

    global_part_coms = []
    current_top = 0

    for i in range(len(stack_ids)):
        part_height = part_heights[i]
        part_local_com = part_axial_coms[i]

        local_top_y = part_height / 2
        global_part_com = current_top + (local_top_y - part_local_com)

        global_part_coms.append(global_part_com)
        current_top += part_height

    total_mass = sum(part_masses)
    if total_mass <= 0:
        return None

    axial_center_of_mass = sum(
        part_mass * global_com for part_mass, global_com in zip(part_masses, global_part_coms)
    ) / total_mass

    center_of_mass_height_fraction = axial_center_of_mass / total_height

    return {
      "stack_ids": stack_ids,
      "part_heights": part_heights,
      "part_diameters": part_diameters,
      "part_masses": part_masses,
      "part_axial_coms": part_axial_coms,
      "total_height": total_height,
      "diameter_transitions": diameter_transitions,
      "diameter_transition_count": diameter_transition_count,
      "diameter_transition_severity": diameter_transition_severity,
      "axial_center_of_mass": axial_center_of_mass,
      "center_of_mass_height_fraction": center_of_mass_height_fraction,
      "slenderness_ratio": slenderness_ratio
    }

### final wrapper to check geometry
def check_geometry_filter(rocket_dict: dict,
                          parts_dict: dict,
                          resource_lookup: dict,
                          verbose: bool = False):

    metrics = compute_geometry_metrics(rocket_dict, parts_dict, resource_lookup)

    if metrics is None:
        if verbose:
            print("FAIL: could not reconstruct inline geometry")
        return False

    failed_rules = []

    if metrics["diameter_transition_count"] > 2.:
        failed_rules.append("too many diameter transitions")

    if metrics["diameter_transition_severity"] > 2.5:
        failed_rules.append("severe diameter transitions")

    if metrics["slenderness_ratio"] > 9.:
        failed_rules.append("extreme slenderness")

    if metrics["center_of_mass_height_fraction"] < 0.2:
        failed_rules.append("top-heavy mass distribution")

    if verbose:
        print("geometry metrics:", metrics)
        print("failed geometry rules:", failed_rules)

    if len(failed_rules) >= 2:
        if verbose:
            print("FAIL: geometry filter")
        return False

    return True


### update validator function

def validate_rocket(rocket_dict: dict,
                    parts_dict: dict,
                    verbose: bool = False):
    for part in rocket_dict['parts']:
        if not check_part_call(part['type'], parts_dict):
            if verbose:
                print(f"FAIL: unknown part type '{part['type']}")
            return False
    geometry_check = check_geometry_filter(rocket_dict, parts_dict, resource_lookup, verbose = verbose)
    if not geometry_check:
        if verbose:
            print('Failed geometry check')
        return False

    checks = [
          (has_minimal_structure(rocket_dict, parts_dict),  "missing root, command, or engine"),
          (check_graph_connections(rocket_dict, parts_dict), "part tree disconnected or has cycles"),
          (check_propellant(rocket_dict, parts_dict),        "propellant incompatibility"),
          (check_staging(rocket_dict),                       "invalid stage references"),
          (check_valid_nodes(rocket_dict, parts_dict),       "invalid attachment nodes"),
      ]
    for result, message in checks:
          if not result:
              if verbose:
                  print(f"FAIL: {message}")
              return False
    return True

check_geometry_filter(rocket_example, parts_by_name, resource_lookup)